# Mixed GAM quickstart
This notebook shows how to train `MixedGAMRegressor`, inspect basis/residual contributions, and launch a Ray Tune search.


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_regression

repo_root = pathlib.Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.models.mixed_gam import MixedGAMRegressor


In [ ]:
X, y = make_regression(
    n_samples=2000,
    n_features=12,
    n_informative=8,
    noise=8.5,
    random_state=42,
)
rng = np.random.default_rng(42)
groups = rng.choice(['north', 'south', 'east', 'west'], size=len(y))


In [ ]:
model = MixedGAMRegressor(mode='group_residual', verbose=False)
model.fit(X, y, groups=groups)

preds, parts = model.predict_and_contrib(X[:5], groups=groups[:5])
print('Predictions:', np.round(preds, 3))
print('Bias term:', parts['bias'][0])
print('Base contributions shape:', parts['base_per_feature'].shape)
print('Residual contributions shape:', parts['residual_per_feature'].shape)


In [ ]:
_ = model.plot_feature_shapes(feature=0, groups=['north', 'south', 'west'])
plt.show()


## Ray Tune search
The cell below launches Ray Tune with an ASHA scheduler. Install Ray first via `pip install "ray[tune]"`.


In [ ]:
from ray import tune

search_result = MixedGAMRegressor.ray_tune_search(
    X,
    y,
    groups=groups,
    num_samples=4,
    val_ratio=0.2,
    max_epochs=120,
    resources_per_trial={'cpu': 2, 'gpu': 0},
    param_space={
        'learning_rate': tune.loguniform(1e-4, 3e-3),
        'base_hidden_units': tune.choice([(64, 32), (128, 64)]),
        'residual_hidden_units': tune.choice([(64,), (96,)]),
        'batch_size': tune.choice([64, 128]),
        'n_epochs': tune.choice([80, 120]),
        'residual_contribution_l1': tune.choice([0.0, 1e-4]),
    },
)

best_cfg = search_result['best_config']
best_cfg
